In [1]:
import pandas as pd
import numpy as np
import openml
from sklearn.preprocessing import StandardScaler,RobustScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [2]:
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
task_ids = benchmark_suite.tasks  # 51 task IDs

task = openml.tasks.get_task(task_ids[0])
dataset = task.get_dataset()
X, y, _, _ = dataset.get_data(target=task.target_name, dataset_format="dataframe")
task_ids

[363612,
 363613,
 363614,
 363615,
 363616,
 363618,
 363619,
 363620,
 363621,
 363623,
 363624,
 363625,
 363626,
 363627,
 363628,
 363629,
 363630,
 363631,
 363632,
 363671,
 363672,
 363673,
 363674,
 363675,
 363676,
 363677,
 363678,
 363679,
 363681,
 363682,
 363683,
 363684,
 363685,
 363686,
 363689,
 363691,
 363693,
 363694,
 363696,
 363697,
 363698,
 363699,
 363700,
 363702,
 363704,
 363705,
 363706,
 363707,
 363708,
 363711,
 363712]

In [3]:
dfs=pd.read_csv("/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/data/task_metadata_tabarena51.csv")

characteristics=dfs.loc[dfs["name"].isin(["QSAR_fish_toxicity","QSAR-TID-11","wine_quality","healthcare_insurance_expenses","Another-Dataset-on-used-Fiat-50","miami_housing"]),:][["tid","name","NumberOfFeatures","target_feature","NumberOfFeatures","NumberOfInstances","NumberOfNumericFeatures","NumberOfSymbolicFeatures"]]
table_df_latex=characteristics.to_latex(caption="Dataset characterstics including shape and dimensions.")

with open("df_table.tex","w") as f:
    f.write(table_df_latex)

In [ ]:
table_df_latex

#### testing



In [ ]:
characteristics

#### loading all datasets


In [4]:
#def preprocessing(dataset,):
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
task_ids = benchmark_suite.tasks  # 51 task IDs
  
characteristics
tables = dict()
for id in characteristics["tid"]:
    task = openml.tasks.get_task(id)
    df = task.get_dataset()
    X, y, _, _ = df.get_data(target=task.target_name, dataset_format="dataframe")
    tables.update({df.name: {"X":X,"y":y.values.reshape(-1,1)}})
    

In [ ]:

def PreProcessing(name,data,test = 0.2,random_seed=0):
    
    X = data.get("X","")
    y = data.get("y","")
    
    is_binary = True if name == "QSAR-TID-11" else False
    is_housing = True if "month_sold" in X.columns else False
        
    if is_binary: # for QSAR TID 11 Dataset   
        print("correct detected")
        X = X.loc[:, X.nunique() > 1]
        
    if is_housing: 
        m = X["month_sold"].astype(int) - 1  # Jan=0 ... Dec=11
        X["month_sold_sin"] = np.sin(2 * np.pi * m / 12.0)
        X["month_sold_cos"] = np.cos(2 * np.pi * m / 12.0)
        X = X.drop(columns=["month_sold"])
                

    num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    #split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test,random_state=random_seed)
    
        
    preprocessor = ColumnTransformer([("numeric",StandardScaler(),num_cols),
                             ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)])
    
    yScaler = StandardScaler()
    
    X_train= pd.DataFrame(data=preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
    X_test = preprocessor.transform(X_test)
    
    y_train = yScaler.fit_transform(y_train)
    y_test = yScaler.transform(y_test)
    
    return X_train, X_test, y_train, y_test, yScaler

In [15]:

X_train, X_test, y_train, y_test, yScaler = PreProcessing("miami_housing",data=tables["miami_housing"])
print(X_train)

       numeric__LATITUDE  numeric__LONGITUDE  numeric__LND_SQFOOT  \
0              -0.102374           -1.177888            -0.185221   
1               0.888649            0.996944            -0.433720   
2              -0.871607           -1.298666            -0.102389   
3              -1.830869           -1.074973            -0.765051   
4               0.034518           -0.741696            -0.017568   
...                  ...                 ...                  ...   
11015          -0.090422           -0.495791            -0.185221   
11016          -0.781092           -0.442292             0.028487   
11017           1.576483            1.395603             0.249650   
11018          -0.530280            0.524634             0.468329   
11019           1.600102            0.363332             0.275660   

       numeric__TOT_LVG_AREA  numeric__SPEC_FEAT_VAL  numeric__RAIL_DIST  \
0                   0.113865                1.332157            1.436678   
1                  

In [11]:
tables["miami_housing"].get("X")

,LATITUDE,LONGITUDE,LND_SQFOOT,TOT_LVG_AREA,SPEC_FEAT_VAL,RAIL_DIST,OCEAN_DIST,WATER_DIST,CNTR_DIST,SUBCNTR_DI,HWY_DIST,age,avno60plus,month_sold,structure_quality
0,25.904986,-80.168793,10016.0,1665.0,1036.0,2437.8,15352.9,2685.1,47229.0,43363.4,13492.3,65,0.0,3,2
1,25.906707,-80.180090,8100.0,2785.0,10388.0,5628.4,19112.2,5121.7,47378.9,45171.4,9779.3,26,0.0,12,4
2,25.609195,-80.388475,8925.0,1767.0,5820.0,4702.0,26217.1,12023.3,88603.0,37892.4,3361.5,19,0.0,9,2
3,25.733496,-80.283502,11000.0,2557.0,55020.0,6416.8,13343.0,1482.3,33691.3,9974.3,11428.8,1,0.0,12,5
4,25.714017,-80.296669,11000.0,2112.0,0.0,3300.1,15585.0,5112.7,41051.6,10737.3,6938.4,61,0.0,10,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13771,25.725312,-80.330714,16781.0,2684.0,36302.0,3621.8,27154.6,6230.8,49049.3,14463.6,3517.3,62,0.0,10,4
13772,25.923716,-80.243016,7866.0,1476.0,1428.0,5974.0,40030.6,2745.7,55899.5,55899.5,920.3,52,0.0,6,4
13773,25.876216,-80.310757,7650.0,1378.0,4602.0,9333.1,58104.8,8622.2,53021.3,49068.5,4034.9,56,0.0,3,4
13774,25.559797,-80.354843,3517.0,1420.0,0.0,8324.8,12982.5,1383.4,95174.7,48664.7,2577.9,9,0.0,5,4


In [ ]:
Rng_Ho_Split = 12

for datasetname in tables:
        X_train_HO, X_test_HO, y_train_HO, y_test_HO, yScaler = PreProcessing(datasetname,data=tables[datasetname],random_seed=Rng_Ho_Split) # we do 1 hold-out-split
        # split train in to test and val
        X_train, X_test,y_train,y_test = train_test_split(X_train_HO,y_train_HO,random_state=Rng_Ho_Split)
        #hpo
        #create study
        #initialize child ruins
        #set main seed for sklearn. 
        def objecttive(trial):
            with ml
        

